<a href="https://colab.research.google.com/github/anphucharry-dev/minigrab-app/blob/main/AI_CODE_ORIGINAl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===============================================================
# MINI GRAB - BLIND-DROP DYNAMIC LOCATION EDITION
# Quét địa điểm thực tế xung quanh khách hàng bằng Overpass API
# ===============================================================

!pip install scikit-fuzzy folium geopy requests ipywidgets pytz -q

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import folium
import requests
import random
import re
import datetime
import pytz
from geopy.distance import geodesic
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ---------------------------------------------------------
# 1. HỆ THỐNG TÀI TRỢ LINH ĐỘNG (DYNAMIC SPONSOR SYSTEM)
# ---------------------------------------------------------
SPONSOR_RULES = {
    "cafe": {"sponsor_rate": 0.8, "msg": "☕ Chạy Deadline năng suất nhé! (Đối tác tài trợ 80%)"},
    "food": {"sponsor_rate": 1.0, "msg": "🍲 Chúc bạn một bữa ăn hoành tráng! (Free 100% cuốc xe)"},
    "chill": {"sponsor_rate": 0.5, "msg": "🌳 Hóng gió xả stress thôi! (Hệ thống trợ giá 50%)"}
}

# Dữ liệu dự phòng (Phòng trường hợp API bản đồ thế giới bị nghẽn) - Chỉ tính ở Sài Gòn
FALLBACK_PLACES = {
    "cafe": [("The Coffee House - Sư Vạn Hạnh", (10.7725, 106.6698)), ("Phúc Long - Ngã 6 Phù Đổng", (10.7706, 106.6912)), ("Katinat - Đồng Khởi", (10.7760, 106.7020))],
    "food": [("Haidilao - Landmark 81", (10.7950, 106.7218)), ("Phố Ẩm Thực Vĩnh Khánh", (10.7584, 106.7011)), ("Cơm Tấm Ba Ghiền", (10.7925, 106.6614))],
    "chill": [("Công viên Bến Bạch Đằng", (10.7735, 106.7082)), ("Phố đi bộ Nguyễn Huệ", (10.7738, 106.7032)), ("Hồ Con Rùa", (10.7828, 106.6960))]
}

def find_nearby_sponsor(lat, lon, intent):
    """Quét các địa điểm thực tế xung quanh bán kính 3000m bằng Overpass API"""
    tags = {
        "cafe": 'node["amenity"="cafe"]',
        "food": 'node["amenity"~"restaurant|fast_food|food_court"]',
        "chill": 'node["leisure"="park"]'
    }

    query = f"""
    [out:json];
    {tags[intent]}(around:3000,{lat},{lon});
    out 15;
    """

    try:
        # Gửi request lấy dữ liệu bản đồ
        url = "http://overpass-api.de/api/interpreter"
        response = requests.get(url, params={'data': query}, timeout=4)
        data = response.json()

        valid_places = []
        for element in data.get('elements', []):
            if 'tags' in element and 'name' in element['tags']:
                valid_places.append((element['tags']['name'], (element['lat'], element['lon'])))

        if valid_places:
            return random.choice(valid_places) # Chọn ngẫu nhiên 1 quán tìm được
    except:
        pass # Nếu lỗi mạng hoặc API chậm, chuyển qua dùng Fallback bên dưới

    # Dùng dữ liệu dự phòng nếu quét thất bại
    return random.choice(FALLBACK_PLACES[intent])

# ---------------------------------------------------------
# 2. PRICING ENGINE - AI FUZZY LOGIC (BÀI 2.11)
# ---------------------------------------------------------
class PricingEngine:
    def __init__(self):
        self.khoang_cach = ctrl.Antecedent(np.arange(0, 51, 1), 'khoang_cach')
        self.luu_luong = ctrl.Antecedent(np.arange(0, 101, 1), 'luu_luong')
        self.nhu_cau = ctrl.Antecedent(np.arange(0, 101, 1), 'nhu_cau')
        self.thoi_tiet = ctrl.Antecedent(np.arange(0, 11, 1), 'thoi_tiet')
        self.danh_gia = ctrl.Antecedent(np.arange(1, 5.1, 0.1), 'danh_gia')
        self.dung_gio = ctrl.Antecedent(np.arange(0, 101, 1), 'dung_gio')

        self.he_so_gia = ctrl.Consequent(np.arange(1, 5.1, 0.1), 'he_so_gia')
        self.thuong = ctrl.Consequent(np.arange(0, 11, 1), 'thuong')

        self.khoang_cach['ngan'] = fuzz.trimf(self.khoang_cach.universe, [0, 0, 3])
        self.khoang_cach['trung_binh'] = fuzz.trimf(self.khoang_cach.universe, [2, 5, 8])
        self.khoang_cach['dai'] = fuzz.trimf(self.khoang_cach.universe, [6, 13, 20])
        self.khoang_cach['rat_xa'] = fuzz.trimf(self.khoang_cach.universe, [15, 50, 50])

        self.luu_luong.automf(names=['thap', 'trung_binh', 'cao'])
        self.nhu_cau.automf(names=['thap', 'trung_binh', 'cao'])
        self.thoi_tiet.automf(names=['tot', 'trung_binh', 'xau'])
        self.danh_gia.automf(names=['kem', 'trung_binh', 'tot'])
        self.dung_gio.automf(names=['tre', 'dung_gio', 'som'])

        self.he_so_gia['thap'] = fuzz.trimf(self.he_so_gia.universe, [1, 1, 2])
        self.he_so_gia['trung_binh'] = fuzz.trimf(self.he_so_gia.universe, [1.5, 2.5, 3.5])
        self.he_so_gia['cao'] = fuzz.trimf(self.he_so_gia.universe, [3, 4, 4.5])
        self.he_so_gia['rat_cao'] = fuzz.trimf(self.he_so_gia.universe, [4, 5, 5])

        self.thuong['khong_co'] = fuzz.trimf(self.thuong.universe, [0, 0, 2])
        self.thuong['it'] = fuzz.trimf(self.thuong.universe, [1, 3, 5])
        self.thuong['trung_binh'] = fuzz.trimf(self.thuong.universe, [4, 6, 8])
        self.thuong['cao'] = fuzz.trimf(self.thuong.universe, [7, 10, 10])

        rules = [
            ctrl.Rule(self.khoang_cach['ngan'] & self.luu_luong['thap'] & self.nhu_cau['thap'], self.he_so_gia['thap']),
            ctrl.Rule(self.khoang_cach['ngan'] & self.luu_luong['trung_binh'] & self.nhu_cau['cao'], self.he_so_gia['trung_binh']),
            ctrl.Rule(self.khoang_cach['trung_binh'] & self.luu_luong['cao'] & self.nhu_cau['cao'], self.he_so_gia['cao']),
            ctrl.Rule(self.khoang_cach['dai'] & self.luu_luong['trung_binh'] & self.thoi_tiet['tot'], self.he_so_gia['trung_binh']),
            ctrl.Rule(self.khoang_cach['dai'] & self.luu_luong['cao'] & self.thoi_tiet['xau'], self.he_so_gia['rat_cao']),
            ctrl.Rule(self.khoang_cach['rat_xa'] & self.thoi_tiet['xau'], self.he_so_gia['rat_cao']),
            ctrl.Rule(self.danh_gia['tot'] & self.dung_gio['som'], self.thuong['cao']),
            ctrl.Rule(self.danh_gia['trung_binh'] & self.dung_gio['dung_gio'], self.thuong['trung_binh']),
            ctrl.Rule(self.khoang_cach['dai'] & self.luu_luong['cao'] & self.dung_gio['dung_gio'], self.thuong['cao']),

            ctrl.Rule(self.luu_luong['trung_binh'] & self.nhu_cau['trung_binh'], self.he_so_gia['trung_binh']),
            ctrl.Rule(self.thoi_tiet['trung_binh'], self.he_so_gia['trung_binh']),
            ctrl.Rule(self.danh_gia['trung_binh'], self.thuong['trung_binh'])
        ]

        self.system = ctrl.ControlSystem(rules)
        self.sim = ctrl.ControlSystemSimulation(self.system)
        self.base_fares = {"bike": 6000, "car": 12000}

    def get_dynamic_app_fee(self, current_hour):
        if 22 <= current_hour or current_hour <= 4: return 10000, "Phí dịch vụ ca đêm"
        elif 17 <= current_hour <= 19: return 8000, "Phí dịch vụ giờ cao điểm"
        else: return 5000, "Phí nền tảng"

    def calculate_fuzzy_surge_and_bonus(self, dist, traffic, demand, weather, rating, punctuality):
        self.sim.input['khoang_cach'] = min(dist, 50)
        self.sim.input['luu_luong'] = traffic
        self.sim.input['nhu_cau'] = demand
        self.sim.input['thoi_tiet'] = weather
        self.sim.input['danh_gia'] = max(1.0, min(rating, 5.0))
        self.sim.input['dung_gio'] = punctuality
        self.sim.compute()
        return self.sim.output['he_so_gia'], self.sim.output['thuong']

# ---------------------------------------------------------
# 3. MATCHING ENGINE & LOCATION
# ---------------------------------------------------------
class MatchingEngine:
    def __init__(self):
        self.driver_names = ["Anh Tuấn", "Chú Hùng", "Minh", "Hoàng", "Nam"]

    def generate_drivers(self, center, n=5):
        return [{"id": f"D{i+1}", "name": random.choice(self.driver_names), "rating": round(random.uniform(4.0, 5.0), 2), "coords": (center[0] + random.uniform(-0.012, 0.012), center[1] + random.uniform(-0.012, 0.012))} for i in range(n)]

    def find_best_driver(self, user_coords, drivers):
        best_driver, best_dist = None, float('inf')
        for d in drivers:
            dist = geodesic(user_coords, d["coords"]).km
            if dist < best_dist: best_driver, best_dist = d, dist
        return best_driver, best_dist

class LocationService:
    def geocode_exact(self, address):
        txt = address.strip()
        search_queries = [txt if "hồ chí minh" in txt.lower() else f"{txt}, Hồ Chí Minh", re.sub(r'^\d+[\w/]*\s+', '', txt) + ", Hồ Chí Minh"]
        headers = {'User-Agent': 'MiniGrab_Ultimate/10.0'}
        for query in search_queries:
            try:
                r = requests.get("https://nominatim.openstreetmap.org/search", params={'q': query, 'format': 'json', 'limit': 1}, headers=headers, timeout=3)
                if r.status_code == 200 and len(r.json()) > 0:
                    return (float(r.json()[0]['lat']), float(r.json()[0]['lon']))
            except: pass
        return None

    def get_route_exact(self, start, end):
        url = f"http://router.project-osrm.org/route/v1/driving/{start[1]},{start[0]};{end[1]},{end[0]}?overview=full&geometries=geojson"
        try:
            r = requests.get(url, timeout=4)
            if r.status_code == 200 and r.json().get("code") == "Ok":
                route = [[p[1], p[0]] for p in r.json()["routes"][0]["geometry"]["coordinates"]]
                return route, r.json()["routes"][0]["distance"] / 1000, r.json()["routes"][0]["duration"] / 60
        except: pass
        km = geodesic(start, end).km * 1.2
        return None, km, km * 3

# ---------------------------------------------------------
# 4. APP CONTROLLER & DYNAMIC UI
# ---------------------------------------------------------
class MiniGrabApp:
    def __init__(self):
        self.pricing = PricingEngine()
        self.location = LocationService()
        self.matching = MatchingEngine()
        self.build_ui()

    def build_ui(self):
        title = widgets.HTML("<h2>🚕 MINIGRAB: TÌM ĐIỂM ĐẾN QUANH BẠN</h2><p><i>Hệ thống quét trực tiếp các quán Cafe/Ăn uống xung quanh vị trí của bạn!</i></p>")

        self.mode_toggle = widgets.ToggleButtons(options=['📍 Chuyến xe Truyền thống', '🎁 Blind-Drop (Khám phá)'], description='Chế độ:', button_style='info')
        self.mode_toggle.observe(self.on_mode_change, names='value')

        self.pickup = widgets.Text(placeholder="Ví dụ: 279 Nguyễn Tri Phương", description="📍 Đang ở:", layout=widgets.Layout(width='90%'))
        self.dropoff = widgets.Text(placeholder="Nhập địa chỉ bạn muốn đến", description="🏁 Điểm đến:", layout=widgets.Layout(width='90%'))

        intent_options = [("☕ Thèm một góc Cafe yên tĩnh", "cafe"), ("🍲 Đang đói, muốn ăn ngon", "food"), ("🌳 Buồn chán, muốn đi hóng gió", "chill")]
        self.blind_intent = widgets.Dropdown(options=intent_options, value="cafe", description="Nhu cầu:", layout=widgets.Layout(width='90%', display='none'))
        self.vehicle = widgets.Dropdown(options=[("🛵 Xe máy", "bike"), ("🚗 Ô tô", "car")], value="bike", description="Loại xe:", layout=widgets.Layout(width='90%'))

        env_title = widgets.HTML("<h4 style='color:#0056b3; margin-top:15px;'>Môi trường (AI Đầu vào):</h4>")
        vn_tz = pytz.timezone('Asia/Ho_Chi_Minh')
        self.ui_time = widgets.IntSlider(value=datetime.datetime.now(vn_tz).hour, min=0, max=23, step=1, description='Giờ đặt:', layout=widgets.Layout(width='90%'))
        self.ui_traffic = widgets.IntSlider(value=50, min=0, max=100, step=1, description='Kẹt xe (%):', layout=widgets.Layout(width='90%'))
        self.ui_weather = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Mưa bão (0-10):', layout=widgets.Layout(width='90%'))

        self.btn = widgets.Button(description="🚀 QUÉT ĐỊA ĐIỂM & ĐẶT XE", button_style="success", layout=widgets.Layout(width='50%', height='45px'))
        self.btn.on_click(self.run_booking)

        self.out = widgets.Output()
        self.message_output = widgets.Output()

        col1 = widgets.VBox([self.mode_toggle, self.pickup, self.dropoff, self.blind_intent, self.vehicle])
        col2 = widgets.VBox([env_title, self.ui_time, self.ui_traffic, self.ui_weather])
        display(widgets.VBox([title, widgets.HBox([col1, col2]), widgets.HTML("<hr>"), self.btn, self.out, self.message_output]))

    def on_mode_change(self, change):
        if change['new'] == '🎁 Blind-Drop (Khám phá)':
            self.dropoff.layout.display = 'none'
            self.blind_intent.layout.display = 'flex'
        else:
            self.dropoff.layout.display = 'flex'
            self.blind_intent.layout.display = 'none'

    def run_booking(self, b):
        with self.out:
            clear_output()
            self.btn.description = "ĐANG TÌM QUÁN QUANH BẠN..."
            self.btn.button_style = "warning"
        with self.message_output: clear_output()

        p = self.location.geocode_exact(self.pickup.value)
        if not p:
            with self.message_output: display(HTML("<div style='color:red;'><b>❌ Lỗi:</b> Vui lòng nhập đúng Điểm Đón!</div>"))
            self.btn.description = "🚀 QUÉT ĐỊA ĐIỂM & ĐẶT XE"; self.btn.button_style = "success"; return

        is_blind_drop = self.mode_toggle.value == '🎁 Blind-Drop (Khám phá)'
        sponsor_meta = None

        if is_blind_drop:
            # SỬ DỤNG HÀM QUÉT ĐỊA ĐIỂM ĐỘNG XUNG QUANH TỌA ĐỘ 'p'
            place_name, d_coord = find_nearby_sponsor(p[0], p[1], self.blind_intent.value)
            sponsor_rules = SPONSOR_RULES[self.blind_intent.value]

            # Gộp dữ liệu tên quán tìm được và luật giảm giá
            sponsor_meta = {
                "name": place_name,
                "msg": sponsor_rules["msg"],
                "sponsor_rate": sponsor_rules["sponsor_rate"]
            }
        else:
            d_coord = self.location.geocode_exact(self.dropoff.value)
            if not d_coord:
                with self.message_output: display(HTML("<div style='color:red;'><b>❌ Lỗi:</b> Không tìm thấy Điểm Đến!</div>"))
                self.btn.description = "🚀 QUÉT ĐỊA ĐIỂM & ĐẶT XE"; self.btn.button_style = "success"; return

        route, km, eta = self.location.get_route_exact(p, d_coord)
        drivers = self.matching.generate_drivers(p)
        drv, drv_km = self.matching.find_best_driver(p, drivers)

        sim_punctuality = random.randint(50, 100)
        surge, bonus = self.pricing.calculate_fuzzy_surge_and_bonus(
            dist=km, traffic=self.ui_traffic.value, demand=60, weather=self.ui_weather.value, rating=drv['rating'], punctuality=sim_punctuality
        )

        phi_app, _ = self.pricing.get_dynamic_app_fee(self.ui_time.value)
        base_fare = km * self.pricing.base_fares[self.vehicle.value]
        total_fare = (base_fare * surge) + phi_app

        if is_blind_drop:
            tai_tro = total_fare * sponsor_meta["sponsor_rate"]
            khach_tra = max(0, total_fare - tai_tro)
        else:
            tai_tro = 0
            khach_tra = total_fare

        with self.out:
            self.render_receipt(drv, drv_km, km, eta, is_blind_drop, sponsor_meta, surge, total_fare, tai_tro, khach_tra, bonus)
            self.render_map(p, d_coord, route, drivers, drv, is_blind_drop, sponsor_meta, self.ui_traffic.value > 60)

        self.btn.description = "🚀 ĐẶT LẠI CHUYẾN KHÁC"; self.btn.button_style = "success"

    def render_receipt(self, drv, drv_km, km, eta, is_blind_drop, sponsor_meta, surge, total_fare, tai_tro, khach_tra, bonus):
        surge_color = "#dc3545" if surge > 1.5 else ("#ffc107" if surge > 1.1 else "#28a745")

        if is_blind_drop:
            display(HTML(f"""<div style="background:linear-gradient(135deg, #6f42c1, #e83e8c);padding:20px;border-radius:10px;margin-bottom:15px;font-family:sans-serif;color:white;box-shadow: 0 4px 15px rgba(0,0,0,0.2);">
                <div style="display:flex; justify-content:space-between; align-items:center;">
                    <h2 style="margin:0 0 10px 0;">🎉 MỞ KHÓA ĐIỂM ĐẾN BÍ MẬT!</h2>
                    <span style="background:rgba(255,255,255,0.2); color:white; padding:5px 10px; border-radius:15px; font-weight:bold; font-size:12px;">Surge Kẹt Xe: {surge:.2f}x</span>
                </div>
                <div style="background:rgba(255,255,255,0.2); padding:15px; border-radius:8px; text-align:center; margin-bottom:15px;">
                    Điểm đến của bạn là: <h3 style="margin:5px 0; color:#fff176; font-size:24px;">{sponsor_meta['name']}</h3>
                    <i>"{sponsor_meta['msg']}"</i>
                </div>
                <div style="display:flex;justify-content:space-between; background:white; color:#333; padding:15px; border-radius:8px;">
                    <div style="width:48%;">
                        👤 <b>{drv['name']}</b> đang tới đón bạn.<br>
                        🎁 Thưởng tài xế (AI tính): <b style="color:#17a2b8;">+{bonus:.1f} đ</b><br>
                        🗺️ Quãng đường: {km:.2f} km
                    </div>
                    <div style="width:48%; text-align:right;">
                        <span style="color:#666; text-decoration:line-through;">Cước gốc: {total_fare:,.0f} ₫</span><br>
                        <span style="color:#28a745; font-weight:bold;">Tài trợ Đối tác: -{tai_tro:,.0f} ₫</span><br>
                        <div style="margin-top:5px; font-size:16px;">BẠN TRẢ: <b style="font-size:24px;color:#dc3545;">{khach_tra:,.0f} ₫</b></div>
                    </div>
                </div></div>"""))
        else:
            display(HTML(f"""<div style="background:#f8f9fa;padding:20px;border-radius:10px;border:2px solid #28a745;margin-bottom:15px;font-family:sans-serif;color:#212529 !important;">
                <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:15px;">
                    <h3 style="color:#28a745;margin:0;">✅ ĐẶT XE THÀNH CÔNG</h3>
                    <span style="background:{surge_color}; color:white; padding:5px 10px; border-radius:15px; font-weight:bold; font-size:14px;">Surge: {surge:.2f}x</span>
                </div>
                <div style="display:flex;justify-content:space-between; background:#fff; padding:15px; border-radius:8px; border:1px solid #e9ecef;">
                    <div style="width:48%;">
                        👤 <b>{drv['name']}</b> ({drv['rating']}⭐)<br>
                        🎁 Thưởng tài xế (AI tính): <b style="color:#17a2b8;">+{bonus:.1f} đ</b>
                    </div>
                    <div style="width:48%; border-left:1px solid #e9ecef; padding-left:15px;">
                        🗺️ Quãng đường: <b>{km:.2f} km</b><br>⏳ Thời gian: <b>{eta:.0f} phút</b>
                    </div>
                </div>
                <div style="text-align:right; margin-top:15px;">
                    <div style="font-size:18px;">TỔNG TIỀN: <b style="font-size:28px;color:#dc3545;">{khach_tra:,.0f} ₫</b></div>
                </div></div>"""))

    def render_map(self, p, d, route, drivers, assigned_drv, is_blind_drop, sponsor_meta, traffic_jam):
        m = folium.Map(location=[(p[0]+d[0])/2, (p[1]+d[1])/2], zoom_start=14)
        folium.Marker(p, popup="Đón", icon=folium.Icon(color="green")).add_to(m)

        d_icon = folium.Icon(color="purple", icon="star") if is_blind_drop else folium.Icon(color="red")
        d_popup = f"Bí mật: {sponsor_meta['name']}" if is_blind_drop else "Đến"
        folium.Marker(d, popup=d_popup, icon=d_icon).add_to(m)

        for x in drivers:
            is_assigned = x["id"] == assigned_drv["id"]
            folium.Marker(x["coords"], icon=folium.Icon(color="blue" if is_assigned else "lightgray", icon="car", prefix='fa')).add_to(m)
        if route:
            folium.PolyLine(route, weight=6, color="red" if traffic_jam else ("#6f42c1" if is_blind_drop else "#007bff"), opacity=0.8).add_to(m)

        # Dùng HTML để ép Google Colab hiển thị bản đồ trong Widget
        display(HTML(m._repr_html_()))

app = MiniGrabApp()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 4.1 MB/s eta 0:00:00
